Q1. 학생 성적 보고서 생성기

In [19]:
# 표준 라이브러리 세 개
import csv # CSV 파일 읽기
import json # JSON 직렬화
import logging # 로그 출력

logging.basicConfig(level=logging.INFO) # INFO 이상 메시지 화면 출력 설정

# 함수 정의
def make_report(csv_path: str, json_path: str) -> int:
    try:
        with open(csv_path, "r", encoding="utf-8") as f: # CSV 파일 열기
            students = list(csv.DictReader(f)) # 각 행을 딕셔너리로 읽고 리스트로 저장
    # 파일이 존재하지 않을 시 WARNING 로고 및 0 반환
    except FileNotFoundError:
        logging.warning(f"CSV 파일이 없습니다: {csv_path}")
        return 0
    # 파일 인코딩이 UTF-8이 아닐 시 ERROR 로고 및 0 반환
    except UnicodeDecodeError:
        logging.error(f"CSV 파일 인코딩이 잘못되었습니다: {csv_path}")
        return 0

    report = [] # 리스트 생성

    for s in students: # CSV 딕셔너리 순회
        mid, fin, hw = s["중간"].strip(), s["기말"].strip(), s["과제"].strip()
        if any(v == "" for v in (mid, fin, hw)): # 세 점수 중 하나라도 빈 문자열이면 결측값
            # 값이 있는 점수는 int, 없는 점수는 None
            scores = {"중간": int(mid) if mid else None,
                      "기말": int(fin) if fin else None,
                      "과제": int(hw)  if hw  else None}
            avg, grade = None, None # 결측값이 있다면 둘 다 None
        else:
            mid, fin, hw = int(mid), int(fin), int(hw) # 세 점수가 모두 있으면 int로 반환
            avg    = mid * 0.3 + fin * 0.5 + hw * 0.2 # 가중 평균
            scores = {"중간": mid, "기말": fin, "과제": hw} # 딕셔너리 저장
            grade  = "A" if avg >= 90 else "B" if avg >= 80 else "C" if avg >= 70 else "F"

        logging.info(f'{s["이름"]}: 평균 {avg}, 등급 {grade}') # 학생 한 명을 처리할 때마다 이름, 평균, 등급을 INFO 로그로 출력
        report.append({"이름": s["이름"], "학번": s["학번"], "점수": scores, "평균": avg, "등급": grade}) # 한 학생의 결과를 딕셔너리로 만들고 리스트에 추가

    # 각 학생의 점수 딕셔너리를 json.dumps로 한 줄 문자열로 변환
    for st in report:
        st["점수"] = json.dumps(st["점수"], ensure_ascii=False)

    output = json.dumps(report, ensure_ascii=False, indent=2) # report 리스트를 JSON 문자열로 변환. 한글 유지. 들여쓰기 적용
    output = output.replace('"점수": "', '"점수": ').replace('\\"', '"').replace('}"', '}') # 따옴표, 이스케이프 제거

    # JSON 문자열 UTF-8로 파일에 저장
    with open(json_path, "w", encoding="utf-8") as f:
        f.write(output)

    # 학생 수 반환
    return len(report)

In [20]:
make_report("scores.csv", "report.json")

INFO:root:김언어: 평균 89.5, 등급 B
INFO:root:이국문: 평균 84.4, 등급 B
INFO:root:박영문: 평균 93.5, 등급 A
INFO:root:최역사: 평균 None, 등급 None


4